# Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

### Built-In middlewares

Provider-agnostic middleware
The following middleware work with any LLM provider:
- Summarization : Automatically summarize conversation history when approaching token limits.
- Human-in-the-loop : Pause execution for human approval of tool calls.
- Model call limit : Limit the number of model calls to prevent excessive costs.
- Tool call limit : Control tool execution by limiting call counts.
- Model fallback : Automatically fallback to alternative models when primary fails.
- PIl detection : Detect and handle Personally Identifiable Information (PII).
- To-do list : Equip agents with task planning and tracking capabilities.
- LLM tool selector : Use an LLM to select relevant tools before calling main model.
- Tool retry : Automatically retry failed tool calls with exponential backoff.
- Model retry : Automatically retry failed model calls with exponential backoff.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")


### Summarisation Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage

# message based summarisation
agent=create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-2.5-flash-lite", #use a smaller model so it costs less
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [5]:
# run with thread id (acts as a user)
config={"configurable":{"thread_id":"test-1"}}

In [6]:
# Alternative test data

questions=[
    "What is 2+2",
    "What is 2*2",
    "What is 2+2/9+7-7-4",
    "What is 7*6",
    "What is 7*7.7",
    "What is 3.9-8.7"
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages : {response}")
    print(f"Messages : {len(response['messages'])}")

Messages : {'messages': [HumanMessage(content='What is 2+2', additional_kwargs={}, response_metadata={}, id='4231d7df-cb3d-44b9-a2bb-ebbf3a71ddce'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7d32-7772-7822-a531-33bb0ddee3ca-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 7, 'total_tokens': 14, 'input_token_details': {'cache_read': 0}})]}
Messages : 2
Messages : {'messages': [HumanMessage(content='What is 2+2', additional_kwargs={}, response_metadata={}, id='4231d7df-cb3d-44b9-a2bb-ebbf3a71ddce'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7d32-7772-7822-a531-33bb0ddee3ca-0', tool_calls=[], invalid_tool_calls=[], u

# LangChain Summarization Middleware

---

## 🎯 Overview

This code demonstrates **LangChain's message-based summarization middleware** that automatically compresses long conversation histories to keep context manageable.

---

## 📦 Key Components

### 1. Agent Setup with SummarizationMiddleware

```python
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-2.5-flash-lite",  # Cheaper model for summarization
            trigger=("messages", 10),                    # Trigger when messages reach 10
            keep=("messages", 4)                         # Keep last 4 messages after summarization
        )
    ]
)

# Run with thread id (acts as a user)
config = {"configurable": {"thread_id": "test-1"}}
```

#### Configuration Parameters

| Parameter | Value | Purpose |
|-----------|-------|---------|
| `model` | `google_genai:gemini-2.5-flash-lite` | Uses smaller/cheaper model for summarization |
| `trigger` | `("messages", 10)` | Activates when total messages reach **10** |
| `keep` | `("messages", 4)` | Retains **last 4 messages** after summarization (2 user + 2 AI) |

---

## 🔄 What Happens During Execution

### Message Count Progression

| Iteration | Question | Total Messages | What Happened |
|-----------|----------|----------------|---------------|
| 1 | `"What is 2+2"` | **2** | No summary (under 10) |
| 2 | `"What is 2*2"` | **4** | No summary (under 10) |
| 3 | `"What is 2+2/9+7-7-4"` | **6** | No summary (under 10) |
| 4 | `"What is 7*6"` | **8** | No summary (under 10) |
| 5 | `"What is 7*7.7"` | **10** | **Triggers at 10!** Summary prepares for next iteration |
| 6 | `"What is 3.9-8.7"` | **6** | **Summary applied!** Reduced from 12→6 messages |

---

## 🎯 The Critical Observation (Last Output)

After the 6th question, instead of having **12 messages** (6 questions + 6 answers), you only have **6 messages**:

```python
Messages: 6  # Reduced from expected 12
```

### Message Structure After Summarization

```python
[
  SystemMessage(
    content="Here is a summary of the conversation to date:\n\n"
            "## SESSION INTENT\n\n"
            "The user is engaged in a series of simple arithmetic calculations.\n\n"
            "## SUMMARY\n\n"
            "The user has asked for the results of several arithmetic operations:\n"
            "*   2 + 2 = 4\n"
            "*   2 * 2 = 4\n"
            "*   2 + 2/9 + 7 - 7 - 4, with the AI clarifying the order of operations...\n"
            "The current query is 'What is 7*6'.\n\n"
            "## ARTIFACTS\n\n"
            "None\n\n"
            "## NEXT STEPS\n\n"
            "Calculate the result of 7 * 6.",
    additional_kwargs={'lc_source': 'summarization'}
  ),  # ← NEW SUMMARY (replaces first 8 messages)
  
  AIMessage(content='7 * 6 = 42', ...),         # ← Last 4 kept (2 AI + 2 Human)
  HumanMessage(content='What is 7*7.7', ...),
  AIMessage(content='7 * 7.7 = 53.9', ...),
  HumanMessage(content='What is 3.9-8.7', ...),
  AIMessage(content='3.9 - 8.7 = -4.8', ...)
]
```

---

## 📝 What the Summary Contains

The generated summary (`SystemMessage`) includes:

### Structure

```markdown
## SESSION INTENT
The user is engaged in a series of simple arithmetic calculations.

## SUMMARY
The user has asked for the results of several arithmetic operations:
*   2 + 2 = 4
*   2 * 2 = 4
*   2 + 2/9 + 7 - 7 - 4, with the AI clarifying the order of operations 
    and providing two possible interpretations, concluding with -1.777... (or -16/9) 
    for the first interpretation.
The current query is "What is 7*6".

## ARTIFACTS
None

## NEXT STEPS
Calculate the result of 7 * 6.
```

### Key Information Preserved

| Section | Content |
|---------|---------|
| **Session Intent** | User is doing simple arithmetic calculations |
| **Summary** | Past questions and answers (2+2, 2*2, complex division) |
| **Current Query** | "What is 7*6" |
| **Next Steps** | "Calculate the result of 7 * 6" |

---

## ⚖️ Why This Matters

### Before vs After Summarization

| Metric | Before Summarization | After Summarization |
|--------|---------------------|---------------------|
| **Message Count** | 12 messages (full history) | 6 messages (summary + recent) |
| **Token Count** | ~1000+ tokens | ~200-300 tokens |
| **Cost** | Higher cost & slower | Lower cost & faster |
| **Risk** | Context window overflow | Safe context usage |

---

## ✅ Key Benefits

1. **Cost Reduction**: Shorter context = fewer tokens = cheaper API calls
2. **Performance**: Faster processing with less data
3. **Context Preservation**: Summary maintains conversation history without storing every message
4. **Automatic**: Happens transparently when threshold is reached
5. **Scalability**: Enables long-running chatbots without context explosion

---

## 🔄 Flow Diagram
### Question 1-5: Accumulate messages (2 → 4 → 6 → 8 → 10)
#                       ↓
### Trigger: 10 messages reached!
#                       ↓
### Question 6 arrives → SummarizationMiddleware activates
#                       ↓
### Summarizes first 8 messages → 1 SystemMessage (summary)
#                       ↓
### Keeps last 4 messages → 4 messages (2 user + 2 AI responses)
#                       ↓
### Total: 1 (summary) + 4 (kept) + 1 (current Q&A) = 6 messages



# Middleware using tokens as triggers

In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage
from langchain_core.tools import tool


@tool
def search_hotels(cuty:str)->str:
    """ Search hotels - returns long responses to use more tokens. """
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi
    """ 

# message based summarisation
agent=create_agent(
    model="groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(), # Saves state to memory
    tools=[search_hotels],
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b", #use a smaller model so it costs less
            trigger=("tokens",550),
            keep=("tokens",200)
        )
    ]
)

config={"configurable":{"thread_id":"test-1"}}
# ↑                    ↑
# |                    └── Unique conversation ID
# └── Required wrapper for configurable params

# Without it: No memory, no context, no summarization
# With it: Agent remembers previous messages in that thread ✅

#Token counter(approx)
def count_tokens(messages):
    total_chars=sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token

In [15]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]}
        , config=config
    )
    tokens = count_tokens (response ["messages" ])
    print(f"{city}: ~{tokens} tokens, {len(response[ 'messages'])} messages")
    print(f"{(response[ 'messages'])}")

Paris: ~138 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='72a485dc-fb78-4e79-af29-d64aef6eca50'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to find hotels in Paris. Let me check the available tools. There\'s a function called search_hotels. The parameters require a "cuty" which is a string. I\'m not sure what "cuty" stands for, but maybe it\'s a typo or shorthand. Since the user is asking for Paris, I\'ll input "Paris" as the value for "cuty". I\'ll proceed with that and generate the tool call.\n', 'tool_calls': [{'id': 'vtn36ym2b', 'function': {'arguments': '{"cuty":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 120, 'prompt_tokens': 157, 'total_tokens': 277, 'completion_time': 0.209235698, 'completion_tokens_details': {'reasoning_tokens': 94}, 'prompt_time': 0.006793781, 'prompt_tokens_details': None, 'queue_

KeyboardInterrupt: 

# Middleware using Fractions as triggers

In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage
from langchain_core.tools import tool


@tool
def search_hotels(cuty:str)->str:
    """ Search hotels"""
    return f"Hotels in {city}: Grand Hotel $350/night,  City Inn $180/night, Budget Stay $75/night" 

# message based summarisation
agent=create_agent(
    model="groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(), # Saves state to memory
    tools=[search_hotels],
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b", #use a smaller model so it costs less
            trigger=("fraction",0.005), # 0.5% = ~640 tokens 
            keep=("fraction",0.002) # 0.2% = ~256 tokens
        )
    ]
)

config={"configurable":{"thread_id":"test-1"}}

#Token counter(approx)
def count_tokens(messages):
    total_chars=sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token

In [17]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]}
        , config=config
    )
    tokens = count_tokens (response ["messages" ])
    fraction=tokens/128000 #groq:qwen/qwen3-32b context window
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} messages")
    print(response[ 'messages'])

Paris: ~93 tokens (0.0727%), 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='b39e9c8d-ba45-4f95-9db1-bbf82340de8f'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to find hotels in Paris. Let me check the available tools. There\'s a function called search_hotels with a parameter "cuty" of type string, which is required. Wait, "cuty" seems like a typo. Maybe it\'s supposed to be "city"? The user mentioned Paris, so I should use that. Even though the parameter name is odd, I\'ll follow the given structure. So, the arguments should be {"cuty": "Paris"}. I\'ll make sure to format the tool call correctly within the XML tags.\n', 'tool_calls': [{'id': 'md6qa4pr8', 'function': {'arguments': '{"cuty":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 142, 'prompt_tokens': 149, 'total_tokens': 291, 'completion_time': 0.261829703, 'comple

### Human In the Loop MiddleWare
Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:
- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [29]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage

def read_email_tool(email_id: str)->str:
    """Mock Function To read email by it's ID"""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient:str,subject:str,body:str)->str:
    """Mock Function To send an email"""
    return f"Email sent to {recipient} with subject '{subject}"



In [30]:
agent=create_agent(
    model="groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(), # Saves state to memory
    tools=[read_email_tool,send_email_tool],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [31]:
config={"configurable":{"thread_id":"test_approve"}}

#Step-1 : Request
result=agent.invoke(
    {"messages":[HumanMessage(content="Send an email to abc@test.com with subject 'Hi' and body 'Karan here'")]},
    config=config
)


Here you will se an "__interrupt__" message, because send_email_tool has intterrupt_on 

In [21]:
result

{'messages': [HumanMessage(content="Send an email to abc@test.com with subject 'Hi' and body 'Karan here'", additional_kwargs={}, response_metadata={}, id='2ae97fbf-575d-4869-af28-8afeab0e99df'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to abc@test.com with the subject 'Hi' and body 'Karan here'. Let me check the available tools. There's the send_email_tool which requires recipient, subject, and body. All three are required. The parameters match what the user provided. So I need to call send_email_tool with those arguments. No need to use the read_email_tool here since the task is about sending, not reading. Just structure the JSON with the correct parameters. Double-check the email, subject, and body to make sure there are no typos. Everything looks good. Ready to output the tool call.\n", 'tool_calls': [{'id': 'z812daf09', 'function': {'arguments': '{"body":"Karan here","recipient":"abc@test.com","subject":"Hi"}', 'name': 

In [22]:
#Step-2 : Approve
from langgraph.types import Command
if "__interrupt__" in result:
    print(" Paused ! Approving ... ")

    result=agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type":"approve"}
                ]
            }
        ),
        config=config
    )
    # print(result)
    print(f"Result : {result['messages'][-1].content}")

 Paused ! Approving ... 
Result : The email has been successfully sent to **abc@test.com** with the subject **"Hi"** and the body **"Karan here"**. Let me know if you need further assistance!


In [23]:
result

{'messages': [HumanMessage(content="Send an email to abc@test.com with subject 'Hi' and body 'Karan here'", additional_kwargs={}, response_metadata={}, id='2ae97fbf-575d-4869-af28-8afeab0e99df'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to abc@test.com with the subject 'Hi' and body 'Karan here'. Let me check the available tools. There's the send_email_tool which requires recipient, subject, and body. All three are required. The parameters match what the user provided. So I need to call send_email_tool with those arguments. No need to use the read_email_tool here since the task is about sending, not reading. Just structure the JSON with the correct parameters. Double-check the email, subject, and body to make sure there are no typos. Everything looks good. Ready to output the tool call.\n", 'tool_calls': [{'id': 'z812daf09', 'function': {'arguments': '{"body":"Karan here","recipient":"abc@test.com","subject":"Hi"}', 'name': 

### Reject

In [ ]:
#Step-2 : Reject
from langgraph.types import Command
if "__interrupt__" in result:
    print(" Paused ! Approving ... ")

    result=agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type":"reject"}
                ]
            }
        ),
        config=config
    )
    # print(result)
    print(f"Result : {result['messages'][-1].content}")

 Paused ! Approving ... 
Result : It seems the email could not be sent at this time. Would you like me to:

1. Attempt to send the email again?
2. Check if there's an existing email in the system that might be conflicting (using read_email_tool)?
3. Modify any details of the email before resending?

Let me know how you'd like to proceed.


### Editing

In [32]:
#Step-2 : Reject
from langgraph.types import Command
if "__interrupt__" in result:
    print(" Paused ! Approving ... ")

    result=agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type":"edit",
                        "edited_action":{
                            "name":"send_email_tool", #tool name
                            "args":{
                                "recipient":"correct@email.com",
                                "subject":"corrected subject",
                                "body":"new body"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    # print(result)
    print(f"Result : {result['messages'][-1].content}")

 Paused ! Approving ... 
Result : The email has been sent to **correct@email.com** with the subject **'corrected subject'** and body **'new body'** instead of your original request. It looks like the details were modified during processing. Would you like me to resend the email with the original parameters (recipient: abc@test.com, subject: 'Hi', body: 'Karan here')?
